### Combining pharm stats for England and Spain

In [1]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from matplotlib.dates import date2num
import pandas as pd

In [ ]:
dfe = pd.read_csv('./data/England_antidepressants_and_anxiolytics_21_25.csv')
dfs = pd.read_csv('./data/Spain_antidepressants_and_anxiolytics_21_25.csv')

In [ ]:
# summing chemical_substance column for group (antidepressants and anxiolytics) per month
dfe_clean = dfe.groupby(['date', 'country', 'group'], as_index=False).agg(
    items_1000=('items_1000', 'sum'),
    items=('items', 'sum')
)
print(dfe_clean.tail(2))
print(dfs.head(2))

In [ ]:
# concating two tables
df_pharm_all = pd.concat([dfe_clean, dfs])
df_pharm_all.head(2)

### Joining Population stats to Pharm stats

In [ ]:
pop = pd.read_csv('./data/Population_monthly_21_25_all.csv')
pop.tail(2)

In [ ]:
# joining tables based on date and country
df_pharm_pop = df_pharm_all.merge(pop, on=['date', 'country'], how='left')
df_pharm_pop.head(2)

In [ ]:
# ensure datetime to avoid wrong calculations 
df_pharm_pop['date'] = pd.to_datetime(df_pharm_pop['date'])

# normalize per 1000 people
df_pharm_pop['items_per_1k'] = round((df_pharm_pop['items'] /(df_pharm_pop['population'])) * 1000,2)

df_pharm_pop.head(4)

### First visualisation
Monthly prescription rates of antidepressants and anxiolytics in England and Spain (2021–2025), including linear trend lines to illustrate long-term prescribing patterns over time.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from matplotlib.dates import date2num

fig, ax = plt.subplots(figsize=(14, 6))

for country, color in zip(
    ['England', 'Spain'],
    ["#2670A6", "#ff5a0e"]
):

    df_c = df_pharm_pop[
        (df_pharm_pop['country'] == country) &
        (df_pharm_pop['group'] == 'Antidepressants')
    ].sort_values('date')

    # Main line
    ax.plot(
        df_c['date'],
        df_c['items_per_1k'],
        label=country,
        color=color,
        linewidth=2
    )

    # Trend line
    x_num = date2num(df_c['date'].values)

    z = np.polyfit(
        x_num,
        df_c['items_per_1k'],
        1
    )

    p = np.poly1d(z)

    ax.plot(
        df_c['date'],
        p(x_num),
        color=color,
        linestyle='--',
        linewidth=1.5,
        alpha=0.7
    )

ax.set_title(
    'Antidepressant Prescriptions per 1,000 Population\nEngland vs Spain (2021–2025)',
    fontsize=14
)

ax.set_xlabel('Date')
ax.set_ylabel('Items per 1,000 Population')

ax.xaxis.set_major_formatter(
    mdates.DateFormatter('%Y-%m')
)

ax.xaxis.set_major_locator(
    mdates.MonthLocator(interval=3)
)

plt.xticks(rotation=45)

ax.legend(title='Country')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

for country, color in zip(['England', 'Spain'], ["#2670A6", "#ff5a0e"]):
    for group, linestyle in zip(['Antidepressants', 'Anxiolytics'], ['-', '--']):
        df_c = df_pharm_pop[(df_pharm_pop['country'] == country) & 
                         (df_pharm_pop['group'] == group)].sort_values('date')
        # Main line
        ax.plot(df_c['date'], df_c['items_per_1k'], 
                label=f'{country} - {group}',
                color=color, linestyle=linestyle, linewidth=2)
        # Trend line
        x_num = date2num(df_c['date'].values)
        z = np.polyfit(x_num, df_c['items_per_1k'], 1)
        p = np.poly1d(z)
        ax.plot(df_c['date'], p(x_num),
                color='red', linestyle='-', linewidth=1.2, alpha=0.5)
ax.set_title('Prescriptions per 1000 people - England vs Spain 2021-2025', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Items per 1000 population')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Ploting next visualisation
From the previous visualistions I saw sum unusual pattern of dips in february. It's most likely becaus of shorter period of days. that is why im addin new column 'items_per_1k_per_day' to normalise it 

In [ ]:
# add days in month
df_pharm_pop['days_in_month'] = df_pharm_pop['date'].dt.days_in_month

# normalize per 1000 people per day
df_pharm_pop['items_per_1k_per_day'] = round((df_pharm_pop['items'] /(df_pharm_pop['population'] * df_pharm_pop['days_in_month'])) * 1000,2)
df_pharm_pop['percent_per_day'] = df_pharm_pop['items_per_1k_per_day'] / 10
df_pharm_pop

In [ ]:
df_pharm_pop.to_csv("Master_table_Spain_England.csv", index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Adding shadowing for summer
years = range(2021, 2026)
for year in years:
    summer_start = pd.Timestamp(f'{year}-06-01')
    summer_end = pd.Timestamp(f'{year}-08-31')
    ax.axvspan(
        summer_start,
        summer_end,
        alpha=0.1,
        color='gold')
    
#  Plot lines
for country, color in zip(
    ['England', 'Spain'],
    ["#2670A6", "#ff5a0e"]
):
    for group, linestyle in zip(
        ['Antidepressants', 'Anxiolytics'],
        ['-', '--']
    ):
        df_c = df_pharm_pop[
            (df_pharm_pop['country'] == country) &
            (df_pharm_pop['group'] == group)
        ].sort_values('date')
        # main line
        ax.plot(
            df_c['date'],
            df_c['items_per_1k_per_day'],
            label=f'{country} - {group}',
            color=color,
            linestyle=linestyle,
            linewidth=2
        )
        # trend line
        x_num = date2num(df_c['date'].values)
        z = np.polyfit(
            x_num,
            df_c['items_per_1k_per_day'],
            1
        )
        p = np.poly1d(z)
        ax.plot(
            df_c['date'],
            p(x_num),
            color='red',
            linestyle='-',
            linewidth=1.2,
            alpha=0.4
        )
ax.set_title(
    'Daily Prescriptions per 1000 People - England vs Spain (2021-2025)',
    fontsize=14
)
ax.set_xlabel('Date')
ax.set_ylabel('Items per 1000 people per day')
ax.xaxis.set_major_formatter(
    mdates.DateFormatter('%Y-%m')
)
ax.xaxis.set_major_locator(
    mdates.MonthLocator(interval=3)
)
plt.xticks(rotation=45)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## __Joining weather data__

In [ ]:
df_w = pd.read_csv('./data/all_weather_clean.csv')
df_w.head(2)

In [ ]:
# checking distinct cities to group
df_w['station_name'].unique()

In [ ]:
# Creating lists of cities for England and Spain 
england_cities = [
    'Sunderland', 'Newcastle upon Tyne', 'Leeds', 'Bristol', 'London',
    'Middlesbrough', 'Manchester', 'York', 'Nottingham', 'Peterborough',
    'Birmingham', 'Plymouth', 'Exeter', 'Brighton', 'Canterbury'
]
spain_cities = [
    'Bilbao', 'Santiago de Compostela', 'Valencia', 'Barcelona',
    'Seville', 'Malaga', 'Madrid', 'Zaragoza'
]
# Add country column
df_w['country'] = df_w['station_name'].apply(
    lambda x: 'England' if x in england_cities else ('Spain' if x in spain_cities else 'Unknown')
)
# Calculate mean, median and std per month per country 

df_weather_agg = df_w.groupby(['date', 'country']).agg(
    tavg_mean=('tavg', 'mean'),
    tavg_std=('tavg', 'std'),
    tmin_mean=('tmin', 'mean'),
    tmin_std=('tmin', 'std'),
    tmax_mean=('tmax', 'mean'),
    tmax_std=('tmax', 'std'),
    prcp_mean=('prcp', 'mean'),
    prcp_std=('prcp', 'std'),
    pres_mean=('pres', 'mean'),
    pres_std=('pres', 'std'),
    tsun_mean=('tsun', 'mean'),
    tsun_std=('tsun', 'std')
).round(1).reset_index()

df_weather_agg.head()

I have a pretty significant standard deviation in the weather data because of outliers.

In [ ]:
df_w[df_w['date'] == '2021-01'][['station_name', 'tsun']].sort_values('station_name').head()

### First visualisation of sun hours 

In [ ]:
# Convert date to datetime for proper x-axis
df_weather_agg['date'] = pd.to_datetime(df_weather_agg['date'])

# convert minutes to hours
df_weather_agg['tsun_mean_hours'] = df_weather_agg['tsun_mean'] / 60
df_weather_agg['tsun_std_hours'] = df_weather_agg['tsun_std'] / 60

fig, ax = plt.subplots(figsize=(14, 6))

for country, color in zip(['England', 'Spain'], ["#5923ed", "#ffb300"]):
    df_c = df_weather_agg[df_weather_agg['country'] == country].sort_values('date')
    
    # Line
    ax.plot(
        df_c['date'],
        df_c['tsun_mean_hours'],
        label=country,
        color=color,
        linewidth=2
    )
    
    # Std band
    ax.fill_between(
        df_c['date'],
        df_c['tsun_mean_hours'] - df_c['tsun_std_hours'],
        df_c['tsun_mean_hours'] + df_c['tsun_std_hours'],
        alpha=0.2,
        color=color,
        label=f'{country} ±1 std'
    )

ax.set_title(
    'Monthly Sunshine Duration (tsun) - England vs Spain 2021-2025',
    fontsize=14
)

ax.set_xlabel('Date')
ax.set_ylabel('Sunshine (hours)')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))

plt.xticks(rotation=45)

ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

for country, color in zip(['England', 'Spain'], ['#1f77b4', '#ff7f0e']):
    df_c = df_weather_agg[df_weather_agg['country'] == country].sort_values('date')
    
    ax.plot(df_c['date'], df_c['prcp_mean'], label=country, color=color, linewidth=2)
    
    ax.fill_between(
        df_c['date'],
        df_c['prcp_mean'] - df_c['prcp_std'],
        df_c['prcp_mean'] + df_c['prcp_std'],
        alpha=0.2,
        color=color,
        label=f'{country} ±1 std'
    )

ax.set_title('Monthly Precipitation (prcp) - England vs Spain 2021-2025', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Precipitation (mm)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
england_cities = [
    'Sunderland', 'Newcastle upon Tyne', 'Leeds', 'Bristol', 'London',
    'Middlesbrough', 'Manchester', 'York', 'Nottingham', 'Peterborough',
    'Birmingham', 'Plymouth', 'Exeter', 'Brighton', 'Canterbury'
]

spain_cities = [
    'Bilbao', 'Santiago de Compostela', 'Valencia', 'Barcelona',
    'Seville', 'Malaga', 'Madrid', 'Zaragoza'
]

# Add country column
df_w['country'] = df_w['station_name'].apply(
    lambda x: 'England' if x in england_cities else ('Spain' if x in spain_cities else 'Unknown')
)

# Calculate average and std per month per country
df_weather_agg_mean_median = df_w.groupby(['date', 'country']).agg(
    tavg_mean=('tavg', 'mean'),
    tavg_median=('tavg', 'median'),
    tavg_std=('tavg', 'std'),
    tmin_mean=('tmin', 'mean'),
    tmin_median=('tmin', 'median'),
    tmin_std=('tmin', 'std'),
    tmax_mean=('tmax', 'mean'),
    tmax_median=('tmax', 'median'),
    tmax_std=('tmax', 'std'),
    prcp_mean=('prcp', 'mean'),
    prcp_median=('prcp', 'median'),
    prcp_std=('prcp', 'std'),
    tsun_mean=('tsun', 'mean'),
    tsun_median=('tsun', 'median'),
    tsun_std=('tsun', 'std')
).round(1).reset_index()


df_weather_agg_mean_median.head(4)

In [ ]:
df_weather_agg_mean_median['date'] = pd.to_datetime(df_weather_agg_mean_median['date'])

fig, ax = plt.subplots(figsize=(14, 6))

for country, color in zip(['England', 'Spain'], ["#5923ed", "#ffb300"]):
    df_c = df_weather_agg_mean_median[df_weather_agg_mean_median['country'] == country].sort_values('date')
    
    ax.plot(df_c['date'], df_c['tsun_median'], label=country, color=color, linewidth=2)

ax.set_title('Monthly Sunshine Duration (tsun) Median - England vs Spain 2021-2025', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Sunshine (minutes)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
df_all = df_pharm_pop.merge(df_weather_agg_mean_median, on=['date', 'country'], how='left')

In [ ]:
df_all.head(2)

In [ ]:
df_all.to_csv("Master_table_Spain_England.csv", index=False)

## Checking the correlation between hours of sunshine and antidepressant prescriptions with a one-month lag.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for country, color in zip(['England', 'Spain'], ["#5923ed", "#ffb300"]):
    df_c = df_all[(df_all['country'] == country) & 
                  (df_all['group'] == 'Antidepressants')].sort_values('date').copy()
    
    # Shift prescriptions by 1 month (lag)
    df_c['items_per_1k_lagged'] = df_c['items_per_1k_per_day'].shift(1)
    df_c = df_c.dropna()  # drop first row which becomes NaN
    
    ax.scatter(df_c['tsun_median'], df_c['items_per_1k_per_day'],
               label=country, color=color, alpha=0.6, s=50)
    
    # Trend line
    z = np.polyfit(df_c['tsun_median'], df_c['items_per_1k_per_day'], 1)
    p = np.poly1d(z)
    x_sorted = df_c['tsun_median'].sort_values()
    ax.plot(x_sorted, p(x_sorted), color=color, linewidth=2)

ax.set_xlabel('Sunshine Duration Median (minutes)')
ax.set_ylabel('Antidepressant Items per 1000 population per day')
ax.set_title('Antidepressant Prescriptions (lagged 1 month) vs Sunshine', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()